## Cell 1: Setup
**What this demonstrates**: Install LangChain's `ParentDocumentRetriever` and supporting stores, load API keys, and configure parent/child chunk sizes — the two parameters that define the retrieval/generation split.
**Expected output**: ✓ Setup complete with model names, chunk sizes, and masked key suffixes.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# All components are in the standard LangChain stack — no extra packages needed
%pip install -q langchain langchain-openai langchain-community chromadb python-dotenv

# ── Standard library ─────────────────────────────────────────────────────────
import os
import time
import pathlib
from typing import Any

# ── Third-party ──────────────────────────────────────────────────────────────
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.stores import InMemoryByteStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ParentDocumentRetriever

# ── Load API keys ─────────────────────────────────────────────────────────────
load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

# ── Constants ─────────────────────────────────────────────────────────────────
LLM_MODEL   = 'claude-sonnet-4-6'
EMBED_MODEL = 'text-embedding-3-small'
CHROMA_DIR  = './chroma_parent_doc'
PARENT_SIZE = 2000   # chars per parent chunk — full regulatory article
CHILD_SIZE  = 400    # chars per child chunk — embedded for search precision

# ── Client initialisation ─────────────────────────────────────────────────────
client:     Anthropic        = Anthropic()            # reads ANTHROPIC_API_KEY from env
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('✓ Setup complete')
print(f'  LLM model:     {LLM_MODEL}')
print(f'  Embed model:   {EMBED_MODEL}')
print(f'  Parent size:   {PARENT_SIZE} chars  (full section — returned to LLM)')
print(f'  Child size:    {CHILD_SIZE} chars   (embedded — used for search only)')
print(f'  Anthropic key: ...{_anthropic_key[-4:]}')
print(f'  OpenAI key:    ...{_openai_key[-4:]}')

## Cell 2: Data — Basel III Regulatory Excerpt
**What this demonstrates**: Load the Basel III document and explain why its dense multi-article structure makes it the ideal test case — a single 400-char chunk of Article 3 gives the leverage ratio threshold but misses the G-SIB buffer condition in Article 3.3.
**Expected output**: Document name, size, first 200 characters, and a clear statement of the flat-chunking failure mode this pattern fixes.

In [ ]:
# ── Locate the Basel III document ─────────────────────────────────────────────
_candidates = [
    pathlib.Path('../../shared/sample_data/basel_iii_excerpt.txt'),
    pathlib.Path('shared/sample_data/basel_iii_excerpt.txt'),
]
SAMPLE_DATA = next((p for p in _candidates if p.exists()), None)
assert SAMPLE_DATA is not None, (
    'Cannot find basel_iii_excerpt.txt. '
    'Run from the repo root or from modules/10_parent_document/.'
)

document_text: str = SAMPLE_DATA.read_text(encoding='utf-8')

print(f'Document:  {SAMPLE_DATA.name}')
print(f'Size:      {len(document_text):,} characters  |  ~{len(document_text.split()):,} words')
print()
print('First 200 characters:')
print('-' * 52)
print(document_text[:200])
print('-' * 52)
print()
print('Document structure (parts):')
for line in document_text.splitlines():
    if line.startswith('PART ') or line.startswith('Article '):
        print(f'  {line[:70]}')
print()
print('Why this document?')
print('  Basel III has 6 structured Parts with numbered Articles and sub-clauses.')
print('  Problem with flat 400-char chunks:')
print('    Query: "explain the leverage ratio" → hits Article 3.1 (the 3.0% minimum)')
print('    But Article 3.3 (G-SIB leverage buffer) is a different chunk — it is')
print('    never retrieved, so the answer is incomplete.')
print('  Parent Document solution:')
print('    Child 3.1 matches the query → parent (full Article 3) is returned')
print('    → LLM receives 3.1 + 3.2 + 3.3 together → complete answer.')

## Cell 3: Core — Parent/Child Splitters and ParentDocumentRetriever
**What this demonstrates**: Build the two-layer index: parent splitter targets full regulatory articles (2000 chars), child splitter creates small embeddable units (400 chars). `ParentDocumentRetriever` wires them together — children go to Chroma, parents to `InMemoryByteStore`.
**Expected output**: Index timing, parent count, child count, and first line of each parent section.

In [ ]:
# ── Parent splitter ───────────────────────────────────────────────────────────
# Targets full regulatory articles (~2000 chars). The '====...' separator aligns
# splits with Basel III section boundaries — avoids cutting mid-article.
SECTION_SEP = '\n================================================================================\n'
parent_splitter = RecursiveCharacterTextSplitter(
    separators=[SECTION_SEP, '\n\n', '\n', ' '],
    chunk_size=PARENT_SIZE,
    chunk_overlap=100,   # 100-char overlap preserves cross-article sentence context
)

# ── Child splitter ────────────────────────────────────────────────────────────
# Small chunks optimised for embedding precision: 400 chars ≈ 2–3 sentences.
# Enough semantic signal to match a query without dilution from surrounding text.
child_splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' '],
    chunk_size=CHILD_SIZE,
    chunk_overlap=50,
)

# ── Vector store: children only ───────────────────────────────────────────────
# Chroma holds child embeddings. The LLM never receives these — they are search
# keys, not context. The embedding_function parameter is required by ParentDocumentRetriever.
vectorstore = Chroma(
    collection_name='parent_doc_children',
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

# ── Doc store: parents only ───────────────────────────────────────────────────
# InMemoryByteStore is a dict-backed store keyed by UUID. Fine for notebooks;
# in production replace with Redis or DynamoDB for persistence across restarts.
docstore = InMemoryByteStore()

# ── Assemble the retriever ─────────────────────────────────────────────────────
# add_documents() does all the work:
#   1. Splits full doc into parents, assigns each a UUID, writes to docstore
#   2. Splits each parent into children, writes UUID into child metadata as 'doc_id'
#   3. Embeds all children and inserts into vectorstore
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# ── Index the document ────────────────────────────────────────────────────────
t0 = time.perf_counter()
full_doc = Document(page_content=document_text, metadata={'source': SAMPLE_DATA.name})
retriever.add_documents([full_doc])
index_ms = (time.perf_counter() - t0) * 1000

n_children = vectorstore._collection.count()

# ── Build parent lookup for Cell 5 inspection ────────────────────────────────
# Reproduce the parent split independently — avoids relying on docstore
# serialisation internals. Every child text is a substring of exactly one parent,
# so we can match child → parent by checking substring containment.
parent_docs_inspect: list[Document] = parent_splitter.create_documents(
    [document_text], metadatas=[{'source': SAMPLE_DATA.name}]
)
parent_texts: list[str] = [p.page_content for p in parent_docs_inspect]

def find_parent_for(child_text: str) -> str | None:
    """Return the parent text that contains this child chunk.

    Children are created by splitting parents, so each child text is a verbatim
    substring of exactly one parent. We match on the first 80 chars to avoid
    false positives from overlap regions at chunk boundaries.

    Args:
        child_text: Raw text of a child chunk from the vector store.

    Returns:
        Parent text string, or None if not found.
    """
    key = child_text.strip()[:80]
    for parent_text in parent_texts:
        if key in parent_text:
            return parent_text
    return None


print(f'Index time:           {index_ms:.0f} ms')
print(f'Parents stored:       {len(parent_texts)} sections  (avg {sum(len(p) for p in parent_texts)//len(parent_texts):,} chars each)')
print(f'Children indexed:     {n_children} chunks    (avg ~{CHILD_SIZE} chars each)')
print(f'Avg children/parent:  {n_children / len(parent_texts):.1f}')
print()
print('Parent sections (first line of each):')
for i, p in enumerate(parent_texts, 1):
    label = next((ln.strip() for ln in p.splitlines() if ln.strip()), '')
    print(f'  [Parent {i:02d}]  {len(p):>5} chars  —  {label[:55]!r}')

## Cell 4: Run — Leverage Ratio Query
**What this demonstrates**: The full Parent Document pipeline: child vector search finds the best-matching sub-clause, `ParentDocumentRetriever` transparently returns the full parent section, and the LLM generates an answer from complete article context.
**Expected output**: Which parent section(s) were returned, a complete answer citing Articles 3.1–3.3, and latency breakdown.

In [ ]:
QUERY = 'Explain the leverage ratio requirement'

# ── Retrieve: ParentDocumentRetriever returns parents, not children ────────────
# Internally: embed query → search child vectorstore → lookup parent by doc_id
# The caller only sees the parent documents — child chunks are invisible.
t0 = time.perf_counter()
parent_docs_returned: list[Document] = retriever.invoke(QUERY)
retrieval_ms = (time.perf_counter() - t0) * 1000

# ── Assemble context from returned parents ─────────────────────────────────────
context = '\n\n---\n\n'.join(
    f'[Regulatory Section {i+1}]\n{doc.page_content}'
    for i, doc in enumerate(parent_docs_returned)
)

SYSTEM = (
    'You are a Basel III regulatory compliance specialist. '
    'Answer using ONLY the provided regulatory excerpts. '
    'Cite specific article numbers (e.g. Article 3.1) and quote numeric thresholds verbatim. '
    'If the context is insufficient, say so explicitly.'
)

# ── Generate ──────────────────────────────────────────────────────────────────
t1 = time.perf_counter()
response = client.messages.create(
    model=LLM_MODEL,
    max_tokens=512,
    system=SYSTEM,
    messages=[{
        'role': 'user',
        'content': f'Basel III excerpts:\n{context}\n\nQuestion: {QUERY}'
    }],
)
answer: str = response.content[0].text
generation_ms = (time.perf_counter() - t1) * 1000

# ── Print results ──────────────────────────────────────────────────────────────
print(f'Query: {QUERY!r}')
print()
print(f'Parent section(s) returned: {len(parent_docs_returned)}')
for i, doc in enumerate(parent_docs_returned, 1):
    label = next((ln.strip() for ln in doc.page_content.splitlines() if ln.strip()), '')
    print(f'  [Section {i}]  {len(doc.page_content):,} chars  —  {label[:55]!r}')
print()
print('=' * 65)
print('ANSWER')
print('=' * 65)
print(answer)
print('=' * 65)
print()
print(f'Retrieval: {retrieval_ms:.0f} ms  |  Generation: {generation_ms:.0f} ms  |  Total: {retrieval_ms + generation_ms:.0f} ms')

## Cell 5: Inspect — Child Chunk vs Parent Context
**What this demonstrates**: Expose the hidden child layer — the actual 400-char chunk that matched the query — and compare it to the full parent section returned to the LLM. Shows the expansion ratio and the specific sub-clause content that flat retrieval would have returned instead.
**Expected output**: Child chunks with similarity scores and parent labels, parent sections with sizes, expansion ratio, and a flat-retrieval comparison showing what would have been missed.

In [ ]:
# ── Direct child search: expose the layer ParentDocumentRetriever hides ────────
# This is what the vector index actually matched — small, high-signal fragments.
# The retriever uses these to find parents but never passes them to the LLM.
child_hits = retriever.vectorstore.similarity_search_with_score(QUERY, k=4)

print(f'Query: {QUERY!r}')
print()
print('── CHILD LAYER  (what the vector index found — 400-char precision chunks) ──')
for i, (child_doc, dist) in enumerate(child_hits, 1):
    # Chroma with cosine space returns distance in [0, 2]; convert to similarity
    sim = 1.0 - dist / 2.0
    parent_text = find_parent_for(child_doc.page_content)
    parent_label = ''
    if parent_text:
        parent_label = next((ln.strip() for ln in parent_text.splitlines() if ln.strip()), '')
    print(f'  [Child {i}]  sim={sim:.3f}  len={len(child_doc.page_content):>3} chars  →  parent: {parent_label[:40]!r}')
    print(f'    {child_doc.page_content[:100].replace(chr(10), " ")!r}...')
    print()

print()
print('── PARENT LAYER  (what the LLM received — full regulatory articles) ────────')
for i, doc in enumerate(parent_docs_returned, 1):
    label = next((ln.strip() for ln in doc.page_content.splitlines() if ln.strip()), '')
    print(f'  [Parent {i}]  {len(doc.page_content):,} chars  —  {label[:55]!r}')
    # Show first and last 80 chars to illustrate the full section scope
    head = doc.page_content[:80].replace(chr(10), ' ')
    tail = doc.page_content[-80:].replace(chr(10), ' ')
    print(f'    Head: {head!r}...')
    print(f'    Tail: ...{tail!r}')
    print()

# ── Context expansion analysis ─────────────────────────────────────────────────
print('── CONTEXT EXPANSION ANALYSIS ───────────────────────────────────────────────')
top_child_len = len(child_hits[0][0].page_content) if child_hits else 0
top_parent_len = len(parent_docs_returned[0].page_content) if parent_docs_returned else 0
if top_child_len and top_parent_len:
    ratio = top_parent_len / top_child_len
    print(f'  Top child chunk size:    {top_child_len:>4} chars  (high embedding precision)')
    print(f'  Parent section returned: {top_parent_len:>4} chars  ({ratio:.1f}× more context for LLM)')
    print()
    print('  What flat retrieval (child only) gives the LLM:')
    flat_preview = child_hits[0][0].page_content.replace(chr(10), ' ')
    print(f'    {flat_preview[:110]!r}...')
    print()
    print('  What Parent Document gives the LLM:')
    parent_preview = parent_docs_returned[0].page_content.replace(chr(10), ' ')
    print(f'    {parent_preview[:110]!r}...')
    print(f'    ... [{top_parent_len - 110} more chars including G-SIB buffer clause]')

## Cell 6: Fintech — G-SIB Multi-Section Capital Requirement
**What this demonstrates**: A cross-article compliance calculation requiring Article 1 (base CET1 + conservation buffer) and Article 9 (G-SIB surcharge) simultaneously. Flat chunking retrieves one article; Parent Document retrieves both, enabling the LLM to calculate the total 9.0% CET1 requirement correctly.
**Expected output**: Retrieved parent sections (should include both Article 1 and Article 9), a step-by-step capital calculation, and an explanation of why flat retrieval produces an incomplete answer.

In [ ]:
# ── Multi-section G-SIB capital calculation ───────────────────────────────────
# This query intentionally spans two separate articles:
#   Article 1: base CET1 minimum 4.5% + conservation buffer 2.5%
#   Article 9: G-SIB Bucket 3 surcharge = 2.0%
# Expected answer: 4.5% + 2.5% + 2.0% = 9.0% total CET1
# A single 400-char flat chunk cannot span both articles — one will be missing.
FINTECH_QUERY = (
    'What is the total CET1 capital requirement for a G-SIB bank in bucket 3, '
    'including all buffers and surcharges?'
)

FINTECH_SYSTEM = (
    'You are a Basel III regulatory capital analyst. '
    'Answer using ONLY the provided regulatory excerpts. '
    'Show the full calculation: base minimum + conservation buffer + G-SIB surcharge = total. '
    'Cite the Article number for each component. '
    'Quote exact percentages verbatim from the source.'
)

# ── Retrieve: expect parents from both Article 1 and Article 9 ────────────────
t0 = time.perf_counter()
gsib_parents: list[Document] = retriever.invoke(FINTECH_QUERY)
retrieval_ms = (time.perf_counter() - t0) * 1000

print(f'Query: {FINTECH_QUERY!r}')
print()
print(f'Parent section(s) returned: {len(gsib_parents)}')
for i, doc in enumerate(gsib_parents, 1):
    label = next((ln.strip() for ln in doc.page_content.splitlines() if ln.strip()), '')
    print(f'  [Section {i}]  {len(doc.page_content):,} chars  —  {label[:55]!r}')

# ── Flat retrieval comparison (what naive RAG would return) ────────────────────
print()
print('Flat retrieval (child only) would return:')
flat_hits = retriever.vectorstore.similarity_search_with_score(FINTECH_QUERY, k=3)
for i, (doc, dist) in enumerate(flat_hits, 1):
    sim = 1.0 - dist / 2.0
    print(f'  [{i}] sim={sim:.3f}  {doc.page_content[:80].replace(chr(10), " ")!r}...')

# ── Generate compliance calculation ───────────────────────────────────────────
context_gsib = '\n\n---\n\n'.join(
    f'[Basel III Section {i+1}]\n{doc.page_content}'
    for i, doc in enumerate(gsib_parents)
)

t1 = time.perf_counter()
response_gsib = client.messages.create(
    model=LLM_MODEL,
    max_tokens=512,
    system=FINTECH_SYSTEM,
    messages=[{
        'role': 'user',
        'content': f'Basel III excerpts:\n{context_gsib}\n\nQuestion: {FINTECH_QUERY}'
    }],
)
gen_ms = (time.perf_counter() - t1) * 1000
answer_gsib: str = response_gsib.content[0].text

print()
print('=' * 65)
print('REGULATORY CAPITAL CALCULATION — G-SIB Bucket 3')
print('=' * 65)
print(answer_gsib)
print('=' * 65)

# ── Why Parent Document matters for this query ────────────────────────────────
print()
print('Cross-article synthesis — why this query requires Parent Document:')
print()
print('  Component          Article   Value')
print('  ─────────────────  ───────   ─────')
print('  CET1 minimum       Art. 1.1  4.5%')
print('  Conservation buf.  Art. 1.4  2.5%')
print('  G-SIB Bucket 3     Art. 9.1  2.0%')
print('  ─────────────────────────────────')
print('  Total CET1 req.              9.0%')
print()
print('  Flat 400-char chunk: retrieves either Article 1 OR Article 9 — not both.')
print('  Parent Document:     both full sections in context → complete calculation.')
print()
print(f'Retrieval: {retrieval_ms:.0f} ms  |  Generation: {gen_ms:.0f} ms')